### Spatial Join: Assigning each Crime to a Ward

#### Importing Packages and Loading Data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import geopandas as gpd

In [2]:
# Setting up Project Paths

project_root = Path.cwd().parent
processed_dir = project_root/"data"/"processed"
wards_path = project_root/"data"/"raw"/"Wards_May_2024_Boundaries_UK_BGC.geojson"

In [3]:
# Loading Data Frame

df = pd.read_parquet(processed_dir/"london_crime.parquet")
df.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv


In [4]:
# Loading Ward Boundaries Data Frame

wards = gpd.read_file(wards_path)

wards

,FID,WD24CD,WD24NM,WD24NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
0,1,E05000932,Ainsdale,,330510,412099,-3.05155,53.60084,96eef5db-0192-45d2-98d6-192a94256c3c,"POLYGON ((-3.0141 53.61128, -3.02003 53.60896,..."
1,2,E05000933,Birkdale,,333046,414534,-3.01375,53.62305,317486af-72c8-439d-99fe-02509bc93488,"POLYGON ((-2.99733 53.62689, -3.00073 53.62498..."
2,3,E05000934,Blundellsands,,330806,399928,-3.04438,53.49150,b5907158-77e9-4411-9f23-6c4d8dcc8f7c,"POLYGON ((-3.02573 53.49215, -3.02537 53.49133..."
3,4,E05000935,Cambridge,,334745,419336,-2.98908,53.66642,41c68ac6-ece4-4c72-8103-7abb47ad0207,"POLYGON ((-2.97989 53.68977, -2.97977 53.68967..."
4,5,E05000936,Church,,332033,397634,-3.02539,53.47105,87c554c4-3837-4f25-aca4-ab90046a43af,"POLYGON ((-3.01413 53.47783, -3.01401 53.47515..."
...,...,...,...,...,...,...,...,...,...,...
8391,8392,W05001796,St Arvans,Llanarfan,352206,199469,-2.69284,51.69193,0099541d-e09b-44b8-84f4-4341092078c5,"POLYGON ((-2.66951 51.74276, -2.66953 51.74273..."
8392,8393,W05001797,St Kingsmark,Llangynfarch,352792,194843,-2.68374,51.65038,f76ce135-bde5-4bf8-9bae-3e9dc9f67adb,"POLYGON ((-2.68581 51.66109, -2.68581 51.66109..."
8393,8394,W05001798,Town,Y Dref,350812,212580,-2.71486,51.80968,709f66df-5762-4289-bb43-14f6797eaed3,"POLYGON ((-2.69851 51.81762, -2.7035 51.81481,..."
8394,8395,W05001799,West End,West End,347579,187362,-2.75795,51.58266,e6589545-a4dc-4274-9b2d-8dee2c2cb861,"POLYGON ((-2.75775 51.58818, -2.75863 51.58757..."


In [5]:
wards.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [6]:
# Our crime data is in latitude/longitude degrees: EPSG:4326 and our wards boundaries are also in the same format

#### Spatial Join

In [7]:
# Building point geometry from longitude/latitude
crime_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)
crime_gdf.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,geometry
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10622 51.51828)
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10768 51.51779)
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828)
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828)
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1121 51.51594)


In [8]:
crime_gdf.shape

(2287673, 12)

In [9]:
wards.head()

,FID,WD24CD,WD24NM,WD24NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
0,1,E05000932,Ainsdale,,330510,412099,-3.05155,53.60084,96eef5db-0192-45d2-98d6-192a94256c3c,"POLYGON ((-3.0141 53.61128, -3.02003 53.60896,..."
1,2,E05000933,Birkdale,,333046,414534,-3.01375,53.62305,317486af-72c8-439d-99fe-02509bc93488,"POLYGON ((-2.99733 53.62689, -3.00073 53.62498..."
2,3,E05000934,Blundellsands,,330806,399928,-3.04438,53.49150,b5907158-77e9-4411-9f23-6c4d8dcc8f7c,"POLYGON ((-3.02573 53.49215, -3.02537 53.49133..."
3,4,E05000935,Cambridge,,334745,419336,-2.98908,53.66642,41c68ac6-ece4-4c72-8103-7abb47ad0207,"POLYGON ((-2.97989 53.68977, -2.97977 53.68967..."
4,5,E05000936,Church,,332033,397634,-3.02539,53.47105,87c554c4-3837-4f25-aca4-ab90046a43af,"POLYGON ((-3.01413 53.47783, -3.01401 53.47515..."


In [10]:
wards.shape

(8396, 10)

In [11]:
# Keeping the needed columns
wards_slim = wards[["WD24CD", "WD24NM", "geometry"]]
wards_slim.head()

,WD24CD,WD24NM,geometry
0,E05000932,Ainsdale,"POLYGON ((-3.0141 53.61128, -3.02003 53.60896,..."
1,E05000933,Birkdale,"POLYGON ((-2.99733 53.62689, -3.00073 53.62498..."
2,E05000934,Blundellsands,"POLYGON ((-3.02573 53.49215, -3.02537 53.49133..."
3,E05000935,Cambridge,"POLYGON ((-2.97989 53.68977, -2.97977 53.68967..."
4,E05000936,Church,"POLYGON ((-3.01413 53.47783, -3.01401 53.47515..."


In [12]:
wards_slim.shape

(8396, 3)

In [13]:
# Joining the dataframes
# how="left" so every crime is kept even if it has no ward

rows_before = len(crime_gdf)

joined = gpd.sjoin(
    crime_gdf,
    wards_slim,
    how="left",
    predicate="within"
)

joined.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,geometry,index_right,WD24CD,WD24NM
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10622 51.51828),4560.0,E05013662,Holborn & Covent Garden
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10768 51.51779),1121.0,E05009305,Farringdon Without
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828),4560.0,E05013662,Holborn & Covent Garden
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828),4560.0,E05013662,Holborn & Covent Garden
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1121 51.51594),4560.0,E05013662,Holborn & Covent Garden


In [14]:
joined

(2287673, 15)

In [21]:
total = len(joined)
matched = joined["WD24CD"].notna().sum()
unmatched = joined["WD24CD"].isna().sum()

print(f"Total crimes:        {total}")
print(f"Matched to a ward:   {matched}")
print(f"Unmatched:           {unmatched}")
print(f"Distinct wards hit:  {joined['WD24CD'].nunique()}")

Total crimes:        2287673
Matched to a ward:   2287478
Unmatched:           195
Distinct wards hit:  905


In [22]:
# The join matched crimes to 905 wards but the total number of Greater London wards is 704. We restrict to the 33 official Greater London authorities (32 boroughs + City of London), identified by the ONS `LAD24CD` code prefix `E09`, using the official ONS Ward-to-LAD lookup.

lookup_path = project_root / "data" / "raw" /"wards_to_lad.csv"
wards_lad = pd.read_csv(lookup_path)

wards_lad.head()

,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,ObjectId
0,E05000932,Ainsdale,NaN,E08000014,Sefton,NaN,1
1,E05000933,Birkdale,NaN,E08000014,Sefton,NaN,2
2,E05000934,Blundellsands,NaN,E08000014,Sefton,NaN,3
3,E05000935,Cambridge,NaN,E08000014,Sefton,NaN,4
4,E05000936,Church,NaN,E08000014,Sefton,NaN,5


In [23]:
wards_lad.shape

(8396, 7)

In [24]:
# Greater London
london_lookup = wards_lad[wards_lad["LAD24CD"].str.startswith("E09", na=False)]
london_lookup.head()

,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,ObjectId
854,E05009288,Aldersgate,NaN,E09000001,City of London,NaN,855
855,E05009289,Aldgate,NaN,E09000001,City of London,NaN,856
856,E05009290,Bassishaw,NaN,E09000001,City of London,NaN,857
857,E05009291,Billingsgate,NaN,E09000001,City of London,NaN,858
858,E05009292,Bishopsgate,NaN,E09000001,City of London,NaN,859


In [25]:
london_lookup.shape

(704, 7)

In [26]:
# Building the borough lookup, keyed by ward code
wards_to_borough = london_lookup.set_index("WD24CD")["LAD24NM"]
wards_to_borough.head()

WD24CD
E05009288    City of London
E05009289    City of London
E05009290    City of London
E05009291    City of London
E05009292    City of London
Name: LAD24NM, dtype: str

In [27]:
# Keep only crimes whose ward is an official Greater London ward
joined_london = joined[joined["WD24CD"].isin(london_lookup["WD24CD"])]
joined_london.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,geometry,index_right,WD24CD,WD24NM
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10622 51.51828),4560.0,E05013662,Holborn & Covent Garden
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10768 51.51779),1121.0,E05009305,Farringdon Without
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828),4560.0,E05013662,Holborn & Covent Garden
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828),4560.0,E05013662,Holborn & Covent Garden
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1121 51.51594),4560.0,E05013662,Holborn & Covent Garden


In [28]:
joined_london.shape

(2284462, 15)

In [29]:
# Attach the borough name
joined_london["borough"] = joined_london["WD24CD"].map(wards_to_borough)
joined_london.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,geometry,index_right,WD24CD,WD24NM,borough
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10622 51.51828),4560.0,E05013662,Holborn & Covent Garden,Camden
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.10768 51.51779),1121.0,E05009305,Farringdon Without,City of London
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828),4560.0,E05013662,Holborn & Covent Garden,Camden
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,POINT (-0.1116 51.51828),4560.0,E05013662,Holborn & Covent Garden,Camden
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,POINT (-0.1121 51.51594),4560.0,E05013662,Holborn & Covent Garden,Camden


In [30]:
joined_london.shape

(2284462, 16)

In [31]:
# Dropping and renaming columns
crime_wards = (
    joined_london
    .rename(columns={"WD24CD": "ward_code", "WD24NM": "ward_name"})
    .drop(columns=["geometry", "index_right", ])
)
crime_wards = pd.DataFrame(crime_wards)

out_path = processed_dir / "london_crime_wards.parquet"
crime_wards.to_parquet(out_path, index=False)

In [32]:
crime_wards

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,ward_code,ward_name,borough
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05009305,Farringdon Without,City of London
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2287668,8cfa73aac9ffd2e8f0d597b4b1e529bacc2cc3e82d509a...,2025-12,Metropolitan Police Service,-0.137230,51.486309,On or near Parking Area,E01035722,Westminster 024G,Violence and sexual offences,Under investigation,2025-12-metropolitan-street.csv,E05013803,Pimlico South,Westminster
2287669,4822666edb3d22a5680ec316ce7c5cef0d861d659b3fa2...,2025-12,Metropolitan Police Service,-0.135320,51.487600,On or near St George'S Square,E01035722,Westminster 024G,Violence and sexual offences,Unable to prosecute suspect,2025-12-metropolitan-street.csv,E05013803,Pimlico South,Westminster
2287670,e86e036bb9da06e7cdfa8ffc5caa4d3e5bb8f68d21a24a...,2025-12,Metropolitan Police Service,-0.133538,51.487500,On or near Aylesford Street,E01035722,Westminster 024G,Violence and sexual offences,Action to be taken by another organisation,2025-12-metropolitan-street.csv,E05013803,Pimlico South,Westminster
2287671,f20a67354e3c9415f5e0281f9e084f1dbae4e858ad0b62...,2025-12,Metropolitan Police Service,-0.133538,51.487500,On or near Aylesford Street,E01035722,Westminster 024G,Violence and sexual offences,Unable to prosecute suspect,2025-12-metropolitan-street.csv,E05013803,Pimlico South,Westminster


### Summary

What is a Ward?

London is made up of "Local Authority Districts (LAD)" regions. They are commonly known as BOROUGHS. There are 33 in Greater London: 32 London boroughs plus the City of London. Examples: Camden, Westminster, Hackney.

A Ward is a subdivision of a Borough. These are the units councillors are elected to represent. A borough contains roughly 15–25 wards. Greater London has 704. Examples: Bloomsbury (in Camden), St James's (in Westminster).

If a ward is a polygon, a boundary drawn on the map, like a puzzle and a crime is a dot on the map then "Which ward did this crime happen in?" therefore means "which polygon does this dot fall inside?". Answering
that question 2.28 million times is the spatial join.

The Office for National Statistics (ONS) is the UK government's official statistics body. It publishes the authoritative boundary files for every ward via its Open Geography Portal. Hence, we used ONS data.

Two ONS files were used, both from the release so their ward
codes match:

1. Ward boundaries (GeoJSON) the actual polygon shape of all 8,396 UK
   wards.
2. Ward-to-Local-Authority lookup (CSV): a table mapping each ward to
   its borough. Needed because a ward's code alone does not tell you which
   borough (or country) it is in.

Every ward and borough has an official ONS code. Ward codes look like `E05009288` ; borough (LAD) codes look like `E09000007`. The leading letter is the country (`E` = England), and the digit block identifies the geography type. Every one of the 33 Greater London authorities has a borough code starting `E09` which is our way to identify "is this London?", and it is what the scope filter uses.

What this notebook did so far:

1. Loaded the 2,287,673 cleaned crimes and the 8,396 UK ward polygons.
2. Geographic data is only valid if both datasets describe locations using the same reference system (CRS). The crimes use latitude/longitude degrees (EPSG:4326, the GPS standard). The ward file was confirmed to already be in EPSG:4326, so the two datasets were directly comparable with no conversion needed.
3. Converted each crime into a map point so it could be tested against the ward polygons.
4. Performed the spatial join. For every crime point, found the ward polygon containing it and attached that ward's code and name.
5. 99.99% of crimes matched a ward. Only 195 (0.01%) did not. Their coordinates might sit inside London maybe snapped onto the River Thames or exactly on a boundary line. They were dropped.
6. Restricted scope to Greater London. The join initially matched 905 wards, not 704. The Metropolitan Police records some crimes which might be just outside the Greater London boundary. Using the ONS lookup, only crimes in wards belonging to the 33 official Greater London authorities (borough code `E09`) were kept.
7. Attached the borough name to every crime which is useful for our model and
   the final map.